In [3]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'BBVA', '2206.pdf')

### Pipeline description

In [4]:
from document_reader import PDFReader
from bank_detector import DefaultBankDetector
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

bank_detector = DefaultBankDetector(PDFReader(statement_path))

extracted_words = bank_detector.get_extracted_words()
extracted_words

AttributeError: 'DefaultBankDetector' object has no attribute 'extracted_words'

In [63]:
bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.get_statement_properties()

print(f'{bank} - {statement_type}')

banamex - credit


In [64]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(bank_detector)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

if statement_type == 'credit' and (start_idx is None or end_idx is None):
    print('New format')
    statement_properties = bank_detector.get_statement_properties(new_credit_format= True)
    boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)
    
    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

252 - 641


,page,text,x0,top,x1,bottom
0,1,Fecha,25.510,476.827,53.900,486.827
1,1,Concepto/Giro,68.027,476.827,136.757,486.827
2,1,de,139.657,476.827,150.647,486.827
3,1,Negocio,153.547,476.827,191.297,486.827
4,1,Población,340.139,476.827,386.069,486.827
...,...,...,...,...,...,...
384,2,PASTERIA,81.635,329.749,123.566,338.749
385,2,QRO,125.897,329.749,144.428,338.749
386,2,CCA,167.738,329.749,185.126,338.749
387,2,230713PY5,187.457,329.749,233.168,338.749


In [65]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, bank_detector)

column_delimitation = row_segmenter.delimit_column_positions()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(column_delimitation)
grouped_rows

6.338000000000001
{'columns': ['Fecha', 'Concepto/Giro de Negocio Mensualidad * / Tipos de Cambio', 'Población / RFC Moneda Ext.', 'Otras Divisas', 'Pesos'], 'x0': [25.51, 68.027, 340.139, 445.016, 529.064], 'x1': [53.900000000000006, 222.69700000000006, 398.65999999999997, 479.716, 557.524]}


,row_group,text,words,top,bottom,page
0,0,Fecha Concepto/Giro de Negocio Población / RFC...,"[(Fecha, 25.51, 53.900000000000006), (Concepto...",476.827,486.827,1
1,1,Moneda Ext. Divisas Mensualidad * / Tipos de C...,"[(Moneda, 340.13, 376.41), (Ext., 379.31, 398....",488.678,498.688,1
2,2,"Jun 10 VIVA AEROBUS CIB ANA050518RL1 1,571.50","[(Jun, 25.51, 40.918), (10, 43.249, 51.835), (...",501.526,510.526,1
3,3,"Jun 10 PAGO INTERBANCARIO 12,151.00-","[(Jun, 25.51, 41.863), (10, 44.473, 56.407), (...",512.864,521.864,1
4,4,Jun 12 AKI SAN ANGELO SSF 830912738 185.00,"[(Jun, 25.51, 40.918), (12, 43.249, 51.1780000...",524.202,533.202,1
5,5,Jun 14 F AHORRO MEGA GARCIA L CFC110121742 126.00,"[(Jun, 25.51, 40.918), (14, 43.249, 51.682), (...",535.540,544.540,1
6,6,Jun 14 BP*REST Y CAFE PLA 120807BK0 603.75,"[(Jun, 25.51, 40.918), (14, 43.249, 51.682), (...",546.878,555.878,1
7,7,"Jun 14 MSFT STORE AME 1404027R0 1,749.00","[(Jun, 25.51, 40.918), (14, 43.249, 51.682), (...",558.216,567.216,1
8,8,Jun 16 STR*UBER EATS UPM 191014S31 7.44,"[(Jun, 25.51, 40.918), (16, 43.249, 51.583), (...",569.554,578.554,1
9,9,"Jun 16 STR*UBER EATS UPM 191014S31 2,148.24","[(Jun, 25.51, 40.918), (16, 43.249, 51.583), (...",580.892,589.892,1


In [66]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, bank_detector)

table_reconstructor.get_structured_table()

,Fecha,Concepto/Giro de Negocio Mensualidad * / Tipos de Cambio,Población / RFC Moneda Ext.,Otras Divisas,Pesos
0,Fecha,Concepto/Giro de Negocio Pesos,Población / RFC,Otras,
1,,Mensualidad * / Tipos de Cambio,Moneda Ext.,Divisas,
2,Jun 10,VIVA AEROBUS CIB ANA050518RL1,,,"1,571.50"
3,Jun 10,PAGO INTERBANCARIO,,,"12,151.00-"
4,Jun 12,AKI SAN ANGELO SSF 830912738,,,185.00
5,Jun 14,F AHORRO MEGA GARCIA L CFC110121742,,,126.00
6,Jun 14,BP*REST Y CAFE PLA 120807BK0,,,603.75
7,Jun 14,MSFT STORE AME 1404027R0,,,"1,749.00"
8,Jun 16,STR*UBER EATS UPM 191014S31,,,7.44
9,Jun 16,STR*UBER EATS UPM 191014S31,,,"2,148.24"


In [67]:
df_structured = table_reconstructor.reconstruct_table()
df_structured

,Fecha,Concepto/Giro de Negocio Mensualidad * / Tipos de Cambio,Población / RFC Moneda Ext.,Otras Divisas,Pesos
0,Jun 10,VIVA AEROBUS CIB ANA050518RL1,,,"1,571.50"
1,Jun 10,PAGO INTERBANCARIO,,,"12,151.00-"
2,Jun 12,AKI SAN ANGELO SSF 830912738,,,185.00
3,Jun 14,F AHORRO MEGA GARCIA L CFC110121742,,,126.00
4,Jun 14,BP*REST Y CAFE PLA 120807BK0,,,603.75
5,Jun 14,MSFT STORE AME 1404027R0,,,"1,749.00"
6,Jun 16,STR*UBER EATS UPM 191014S31,,,7.44
7,Jun 16,STR*UBER EATS UPM 191014S31,,,"2,148.24"
8,Jun 16,MERPAGO*HELADOSMAR MAG 2105031W3,,,312.00
9,Jun 16,STRIPE *UBER EATS UPM 191014S31,,,64.67


In [68]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, bank_detector)

df_normalized = normalizer.normalize_table()
df_normalized

,Date,Description,Amount,Type
0,2024-06-10,VIVA AEROBUS CIB ANA050518RL1,-1571.50,Cargo
1,2024-06-10,PAGO INTERBANCARIO,12151.00,Abono
2,2024-06-12,AKI SAN ANGELO SSF 830912738,-185.00,Cargo
3,2024-06-14,F AHORRO MEGA GARCIA L CFC110121742,-126.00,Cargo
4,2024-06-14,BP*REST Y CAFE PLA 120807BK0,-603.75,Cargo
5,2024-06-14,MSFT STORE AME 1404027R0,-1749.00,Cargo
6,2024-06-16,STR*UBER EATS UPM 191014S31,-7.44,Cargo
7,2024-06-16,STR*UBER EATS UPM 191014S31,-2148.24,Cargo
8,2024-06-16,MERPAGO*HELADOSMAR MAG 2105031W3,-312.00,Cargo
9,2024-06-16,STRIPE *UBER EATS UPM 191014S31,-64.67,Cargo


In [69]:
normalizer.period_idx

51

In [70]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_normalized)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_to_csv(file_path)

Data exported to c:\Users\mario\Proyectos\Aether\Aether_v1\documents\outputs\banamex\banamex_credit.csv successfully.
